In [ ]:
# ==============================================================================
# PARALLEL DIM: CAR + DATE
# ==============================================================================
from notebooks.helpers import (
    IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger,
    write_gold_table, safe_count, generate_batch_id,
    generate_date_dimension, generate_role_playing_date_dimensions,
)
from notebooks.helpers.silver_transforms import transform_car_full_pipeline, build_equipment_bridges
from datetime import date
import pandas as pd

logger = setup_logger("parallel_dim_car_date")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

bronze_batch_id = get_latest_batch_id(spark, "inventory")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for inventory; run bronze load first.")
logger.info(f"Using bronze batch_id for inventory: {bronze_batch_id}")

car_config = TableConfig(
    table_name="inventory",
    business_key="inventory_id",
    surrogate_key="car_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_car",
    silver_transform=transform_car_full_pipeline,
    dependencies=["car", "inventory_equipment", "equipment"],
)

ROLE_PLAYING_DATES = [
    ("dim_service_date", "service_date_key"),
    ("dim_rental_date", "rental_date_key"),
    ("dim_return_date", "return_date_key"),
    ("dim_payment_date", "payment_date_key"),
    ("dim_payment_deadline_date", "payment_deadline_date_key"),
]

print("Row Counts (Before):")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"bridge_equipment_group_equipment: {safe_count(spark, 'bridge_equipment_group_equipment')}")
print(f"bridge_car_equipment: {safe_count(spark, 'bridge_car_equipment')}")
print(f"dim_date: {safe_count(spark, 'dim_date')}")
print(f"dim_service_date: {safe_count(spark, 'dim_service_date')}")
print(f"dim_rental_date: {safe_count(spark, 'dim_rental_date')}")
print(f"dim_return_date: {safe_count(spark, 'dim_return_date')}")
print(f"dim_payment_date: {safe_count(spark, 'dim_payment_date')}")
print(f"dim_payment_deadline_date: {safe_count(spark, 'dim_payment_deadline_date')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))


In [ ]:
results = pipeline.load_tables([car_config], force_full=False, bronze_batch_id=bronze_batch_id)

# Rebuild equipment bridges after inventory/equipment load
inventory_equipment_bronze = spark.table("wheelie.bronze.inventory_equipment")
bridge_equipment_group_equipment, bridge_car_equipment = build_equipment_bridges(
    inventory_equipment_bronze
)
write_gold_table(bridge_equipment_group_equipment, "bridge_equipment_group_equipment", mode="overwrite")
write_gold_table(bridge_car_equipment, "bridge_car_equipment", mode="overwrite")

# Build dim_date and role-playing dimensions
if not spark.catalog.tableExists("wheelie.gold.dim_date"):
    logger.info("dim_date missing; generating base date dimension")
    dim_date = generate_date_dimension(
        spark,
        start_date=date(2000, 1, 1),
        end_date=date(2027, 12, 31),
    )
    write_gold_table(dim_date, "dim_date", mode="overwrite")
else:
    dim_date = spark.table("wheelie.gold.dim_date")

missing_role_dims = [
    (table_name, key_name)
    for table_name, key_name in ROLE_PLAYING_DATES
    if not spark.catalog.tableExists(f"wheelie.gold.{table_name}")
]

if missing_role_dims:
    logger.info(f"Creating {len(missing_role_dims)} role-playing date dimensions")
    role_playing_dims = generate_role_playing_date_dimensions(dim_date, missing_role_dims)
    for table_name, df in role_playing_dims:
        write_gold_table(df, table_name, mode="overwrite")
else:
    logger.info("All role-playing date dimensions already exist")

display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_car: {safe_count(spark, 'dim_car')}")
print(f"bridge_equipment_group_equipment: {safe_count(spark, 'bridge_equipment_group_equipment')}")
print(f"bridge_car_equipment: {safe_count(spark, 'bridge_car_equipment')}")
print(f"dim_date: {safe_count(spark, 'dim_date')}")
print(f"dim_service_date: {safe_count(spark, 'dim_service_date')}")
print(f"dim_rental_date: {safe_count(spark, 'dim_rental_date')}")
print(f"dim_return_date: {safe_count(spark, 'dim_return_date')}")
print(f"dim_payment_date: {safe_count(spark, 'dim_payment_date')}")
print(f"dim_payment_deadline_date: {safe_count(spark, 'dim_payment_deadline_date')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
